# TruthLens AI — SciFact Dataset Deep-Dive EDA
This notebook explores the SciFact scientific fact-checking benchmark (EMNLP 2020).

### Topics Covered:
- Corpus document distribution (research paper abstracts, sentence counts)
- Claim verification structure (claims with evidence vs. claims without evidence / NEI)
- Grounded `(claim, evidence_text, label)` triple reconstruction
- Rationale sentence length distributions
- Cross-validation fold consistency
- Cross-split document leakage analysis

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath("../src"))
from scifact_loader import load_scifact_corpus, load_scifact_claims, reconstruct_scifact_triples, get_scifact_statistics

sns.set_theme(style="whitegrid")
corpus = load_scifact_corpus("../data/raw/scifact/corpus.jsonl")
claims_tr = load_scifact_claims("../data/raw/scifact/claims_train.jsonl")
claims_dv = load_scifact_claims("../data/raw/scifact/claims_dev.jsonl")
claims_te = load_scifact_claims("../data/raw/scifact/claims_test.jsonl")

print(f"Corpus: {len(corpus):,} papers")
print(f"Claims: Train={len(claims_tr)}, Dev={len(claims_dv)}, Test={len(claims_te)}")

## 1. Corpus Abstract Statistics

In [ ]:
abstract_sents = [len(d.get("abstract", [])) for d in corpus.values()]
print(f"Total sentences across corpus: {sum(abstract_sents):,}")
print(f"Sentences per abstract: Min={min(abstract_sents)}, Max={max(abstract_sents)}, Mean={np.mean(abstract_sents):.2f}")

plt.figure(figsize=(8, 4))
sns.histplot(abstract_sents, bins=30, range=(1, 25), color="#7570b3")
plt.title("SciFact Corpus: Abstract Sentence Count Distribution")
plt.xlabel("Sentences in Abstract")
plt.tight_layout()
plt.show()

## 2. Claim Evidence Availability

In [ ]:
tr_has_ev = [bool(c.get("evidence", {})) for c in claims_tr]
dv_has_ev = [bool(c.get("evidence", {})) for c in claims_dv]

print(f"Train claims with evidence: {sum(tr_has_ev)} / {len(claims_tr)} ({sum(tr_has_ev)/len(claims_tr)*100:.1f}%)")
print(f"Dev claims with evidence: {sum(dv_has_ev)} / {len(claims_dv)} ({sum(dv_has_ev)/len(claims_dv)*100:.1f}%)")

## 3. Grounded Triples Reconstruction

In [ ]:
triples_tr = reconstruct_scifact_triples(claims_tr, corpus, include_nei=True)
print(f"Total reconstructed triples (train): {len(triples_tr)}")
print("Label counts:")
print(triples_tr["label"].value_counts())

display(triples_tr.head(5))

## 4. Evidence Rationale Sentence Lengths

In [ ]:
triples_tr["ev_sentence_count"] = triples_tr["sentence_indices"].apply(len)
rationale_sub = triples_tr[triples_tr["ev_sentence_count"] > 0]

plt.figure(figsize=(7, 4))
sns.countplot(x=rationale_sub["ev_sentence_count"], palette="Purples_r")
plt.title("Sentences per Evidence Rationale in SciFact")
plt.xlabel("Number of Sentences")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 5. Cross-Split Document Leakage

In [ ]:
tr_docs = set(d for c in claims_tr for d in c.get("cited_doc_ids", []))
dv_docs = set(d for c in claims_dv for d in c.get("cited_doc_ids", []))

print(f"Train unique cited docs: {len(tr_docs)}")
print(f"Dev unique cited docs: {len(dv_docs)}")
overlap_docs = tr_docs.intersection(dv_docs)
print(f"Cited docs appearing in BOTH train and dev: {len(overlap_docs)} ({len(overlap_docs)/len(dv_docs)*100:.1f}% of dev docs)")